<a href="https://colab.research.google.com/github/Josdv0/AgenciaViajesDB/blob/main/_004_memoria_cache/medicion_de_cache_con_perf_en_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Medición de caché con `perf` en C

El objetivo es construir una práctica en C para estudiar la jerarquía de memoria caché y medir, con `perf`, el efecto del patrón de acceso por filas y por columnas sobre L1, L2, L3, IPC, ciclos y tiempo de ejecución.


## 1. Medir solamente el recorrido

La primera modificación propuesta consiste en separar la inicialización de la matriz de la región que se desea medir.

La idea inicial fue:

1. reservar memoria;
2. inicializar la matriz;
3. detener el proceso con `SIGSTOP`;
4. conectar `perf` al PID;
5. reanudar el proceso;
6. ejecutar únicamente el recorrido;
7. terminar.


In [10]:
%%writefile programa.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#include <unistd.h>
#include <signal.h>

#define N 8000

double *A;

double tiempo_segundos(void)
{
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec + t.tv_nsec / 1e9;
}

double recorrer_filas(void)
{
    double suma = 0.0;
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++)
            suma += A[(size_t)i * N + j];
    return suma;
}

double recorrer_columnas(void)
{
    double suma = 0.0;
    for (int j = 0; j < N; j++)
        for (int i = 0; i < N; i++)
            suma += A[(size_t)i * N + j];
    return suma;
}

int main(int argc, char *argv[])
{
    if (argc != 2) return 1;

    size_t elementos = (size_t)N * N;
    size_t bytes = elementos * sizeof(double);

    A = malloc(bytes);
    if (A == NULL) return 1;

    for (size_t i = 0; i < elementos; i++) A[i] = 1.0;

    double inicio = tiempo_segundos();
    double suma = 0;

    if (strcmp(argv[1], "filas") == 0) suma = recorrer_filas();
    else if (strcmp(argv[1], "columnas") == 0) suma = recorrer_columnas();

    double fin = tiempo_segundos();

    printf("Modo: %s | Suma: %.1f | Tiempo: %.6f s\n", argv[1], suma, fin - inicio);

    free(A);
    return 0;
}

Overwriting programa.c


### Compilación

```bash
gcc -O0 cache.c -o cache
```

### Ejecución

En una terminal:

```bash
taskset -c 0 ./cache filas
```

El programa se detiene antes del recorrido y muestra su PID.

En otra terminal:

```bash
perf stat -p PID \
-e cycles,instructions,\
L1-dcache-loads,L1-dcache-load-misses,\
l2_rqsts.demand_data_rd_hit,l2_rqsts.demand_data_rd_miss &
```

Después:

```bash
kill -CONT PID
```

Conceptualmente:

```text
malloc
  ↓
inicialización
  ↓
SIGSTOP
  ↓
perf se conecta
  ↓
SIGCONT
  ↓
recorrido
  ↓
fin
```


## 2. Versión definitiva con `perf_event_open()`

Después se propuso una versión donde el propio programa abre y controla los contadores con `perf_event_open()`. De esta manera, los contadores se habilitan justo antes del recorrido y se deshabilitan justo después.

La primera versión mide:

- `cycles`
- `instructions`
- `L1-dcache-loads`
- `L1-dcache-load-misses`


In [13]:
%%writefile perf_matrices.c
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>
#include <unistd.h>
#include <sys/ioctl.h>
#include <sys/syscall.h>
#include <linux/perf_event.h>
#include <time.h>

#define N 8000

double *A;

static long perf_event_open(
    struct perf_event_attr *hw_event,
    pid_t pid,
    int cpu,
    int group_fd,
    unsigned long flags)
{
    return syscall(
        __NR_perf_event_open,
        hw_event,
        pid,
        cpu,
        group_fd,
        flags
    );
}

int crear_contador(
    uint32_t type,
    uint64_t config,
    int group_fd)
{
    struct perf_event_attr pe;

    memset(&pe, 0, sizeof(pe));

    pe.type = type;
    pe.size = sizeof(pe);
    pe.config = config;
    pe.disabled = 1;
    pe.exclude_kernel = 1;
    pe.exclude_hv = 1;

    int fd = perf_event_open(
        &pe,
        0,
        -1,
        group_fd,
        0
    );

    if (fd == -1)
    {
        perror("perf_event_open");
        exit(EXIT_FAILURE);
    }

    return fd;
}

double tiempo_segundos(void)
{
    struct timespec t;

    clock_gettime(CLOCK_MONOTONIC, &t);

    return (double)t.tv_sec + (double)t.tv_nsec / 1e9;
}

double recorrer_filas(void)
{
    double suma = 0.0;

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            suma += A[(size_t)i * N + j];
        }
    }

    return suma;
}

double recorrer_columnas(void)
{
    double suma = 0.0;

    for (int j = 0; j < N; j++)
    {
        for (int i = 0; i < N; i++)
        {
            suma += A[(size_t)i * N + j];
        }
    }

    return suma;
}

int main(int argc, char *argv[])
{
    if (argc != 2)
    {
        printf("Uso:\n");
        printf("  %s filas\n", argv[0]);
        printf("  %s columnas\n", argv[0]);
        return EXIT_FAILURE;
    }

    size_t elementos = (size_t)N * N;
    size_t bytes = elementos * sizeof(double);

    printf("Matriz: %d x %d\n", N, N);
    printf("Elementos: %zu\n", elementos);
    printf("Memoria: %.2f MiB\n", bytes / (1024.0 * 1024.0));

    A = malloc(bytes);

    if (A == NULL)
    {
        perror("malloc");
        return EXIT_FAILURE;
    }

    for (size_t i = 0; i < elementos; i++)
    {
        A[i] = 1.0;
    }

    int fd_cycles =
        crear_contador(
            PERF_TYPE_HARDWARE,
            PERF_COUNT_HW_CPU_CYCLES,
            -1
        );

    int fd_instructions =
        crear_contador(
            PERF_TYPE_HARDWARE,
            PERF_COUNT_HW_INSTRUCTIONS,
            -1
        );

    uint64_t l1_load_config =
        PERF_COUNT_HW_CACHE_L1D |
        (PERF_COUNT_HW_CACHE_OP_READ << 8) |
        (PERF_COUNT_HW_CACHE_RESULT_ACCESS << 16);

    int fd_l1_loads =
        crear_contador(
            PERF_TYPE_HW_CACHE,
            l1_load_config,
            -1
        );

    uint64_t l1_miss_config =
        PERF_COUNT_HW_CACHE_L1D |
        (PERF_COUNT_HW_CACHE_OP_READ << 8) |
        (PERF_COUNT_HW_CACHE_RESULT_MISS << 16);

    int fd_l1_misses =
        crear_contador(
            PERF_TYPE_HW_CACHE,
            l1_miss_config,
            -1
        );

    ioctl(fd_cycles, PERF_EVENT_IOC_RESET, 0);
    ioctl(fd_instructions, PERF_EVENT_IOC_RESET, 0);
    ioctl(fd_l1_loads, PERF_EVENT_IOC_RESET, 0);
    ioctl(fd_l1_misses, PERF_EVENT_IOC_RESET, 0);

    ioctl(fd_cycles, PERF_EVENT_IOC_ENABLE, 0);
    ioctl(fd_instructions, PERF_EVENT_IOC_ENABLE, 0);
    ioctl(fd_l1_loads, PERF_EVENT_IOC_ENABLE, 0);
    ioctl(fd_l1_misses, PERF_EVENT_IOC_ENABLE, 0);

    double inicio = tiempo_segundos();

    double suma;

    if (strcmp(argv[1], "filas") == 0)
    {
        suma = recorrer_filas();
    }
    else if (strcmp(argv[1], "columnas") == 0)
    {
        suma = recorrer_columnas();
    }
    else
    {
        printf("Modo inválido.\n");
        free(A);
        return EXIT_FAILURE;
    }

    double fin = tiempo_segundos();

    ioctl(fd_cycles, PERF_EVENT_IOC_DISABLE, 0);
    ioctl(fd_instructions, PERF_EVENT_IOC_DISABLE, 0);
    ioctl(fd_l1_loads, PERF_EVENT_IOC_DISABLE, 0);
    ioctl(fd_l1_misses, PERF_EVENT_IOC_DISABLE, 0);

    long long cycles;
    long long instructions;
    long long l1_loads;
    long long l1_misses;

    read(fd_cycles, &cycles, sizeof(cycles));
    read(fd_instructions, &instructions, sizeof(instructions));
    read(fd_l1_loads, &l1_loads, sizeof(l1_loads));
    read(fd_l1_misses, &l1_misses, sizeof(l1_misses));

    double ipc = (double)instructions / cycles;
    double l1_miss_rate = 100.0 * (double)l1_misses / l1_loads;

    printf("\n");
    printf("Modo: %s\n", argv[1]);
    printf("Suma: %.1f\n", suma);
    printf("Tiempo recorrido: %.6f s\n", fin - inicio);

    printf("\n===== PERF =====\n");

    printf("Cycles:              %lld\n", cycles);
    printf("Instructions:        %lld\n", instructions);
    printf("IPC:                 %.3f\n", ipc);
    printf("L1 loads:            %lld\n", l1_loads);
    printf("L1 load misses:      %lld\n", l1_misses);
    printf("L1 miss rate:        %.3f %%\n", l1_miss_rate);

    close(fd_cycles);
    close(fd_instructions);
    close(fd_l1_loads);
    close(fd_l1_misses);

    free(A);

    return EXIT_SUCCESS;
}

Writing perf_matrices.c


### Compilación y ejecución

```bash
gcc -O0 cache_perf.c -o cache_perf
```

```bash
taskset -c 0 ./cache_perf filas
taskset -c 0 ./cache_perf columnas
```

La región medida queda delimitada por:

```text
RESET
  ↓
ENABLE
  ↓
recorrido
  ↓
DISABLE
  ↓
lectura de contadores
```


## 3. Versión para medir L1, L2 y L3 con `perf stat`

Para L2 y L3 se decidió aprovechar los nombres simbólicos de eventos que ya estaban disponibles en la CPU:

```text
L1-dcache-loads
L1-dcache-load-misses

l2_rqsts.demand_data_rd_hit
l2_rqsts.demand_data_rd_miss

mem_load_retired.l3_hit
mem_load_retired.l3_miss
```

En lugar de codificar eventos RAW, el programa usa `prctl()` para habilitar y deshabilitar los eventos heredados de `perf stat` únicamente alrededor del recorrido.


In [16]:
%%writefile programa_prctl.c
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#include <sys/prctl.h>
#include <linux/prctl.h>

#define N 8000

double *A;

double tiempo_segundos(void)
{
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return (double)t.tv_sec + (double)t.tv_nsec / 1e9;
}

double recorrer_filas(void)
{
    double suma = 0.0;
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++)
            suma += A[(size_t)i * N + j];
    return suma;
}

double recorrer_columnas(void)
{
    double suma = 0.0;
    for (int j = 0; j < N; j++)
        for (int i = 0; i < N; i++)
            suma += A[(size_t)i * N + j];
    return suma;
}

int main(int argc, char *argv[])
{
    prctl(PR_TASK_PERF_EVENTS_DISABLE);

    if (argc != 2)
    {
        printf("Uso: %s [filas|columnas]\n", argv[0]);
        return EXIT_FAILURE;
    }

    size_t elementos = (size_t)N * N;
    size_t bytes = elementos * sizeof(double);

    printf("Matriz: %d x %d\n", N, N);
    printf("Elementos: %zu\n", elementos);
    printf("Memoria: %.2f MiB\n", bytes / (1024.0 * 1024.0));

    A = malloc(bytes);
    if (A == NULL)
    {
        perror("malloc");
        return EXIT_FAILURE;
    }

    for (size_t i = 0; i < elementos; i++)
    {
        A[i] = 1.0;
    }

    int modo_filas;
    if (strcmp(argv[1], "filas") == 0)
        modo_filas = 1;
    else if (strcmp(argv[1], "columnas") == 0)
        modo_filas = 0;
    else
    {
        printf("Modo inválido.\n");
        free(A);
        return EXIT_FAILURE;
    }

    double suma, inicio, fin;

    /* Inicio de medición perf */
    prctl(PR_TASK_PERF_EVENTS_ENABLE);
    inicio = tiempo_segundos();

    if (modo_filas)
        suma = recorrer_filas();
    else
        suma = recorrer_columnas();

    fin = tiempo_segundos();
    prctl(PR_TASK_PERF_EVENTS_DISABLE);
    /* Fin de medición perf */

    printf("\nModo: %s\n", argv[1]);
    printf("Suma: %.1f\n", suma);
    printf("Tiempo recorrido: %.6f s\n", fin - inicio);

    free(A);
    return EXIT_SUCCESS;
}

Writing programa_prctl.c


### Compilación

```bash
gcc -O0 cache_perf.c -o cache_perf
```

### Medición simultánea de L1, L2 y L3

Para filas:

```bash
taskset -c 0 perf stat \
-e cycles,instructions,\
L1-dcache-loads,L1-dcache-load-misses,\
l2_rqsts.demand_data_rd_hit,l2_rqsts.demand_data_rd_miss,\
mem_load_retired.l3_hit,mem_load_retired.l3_miss \
./cache_perf filas
```

Para columnas:

```bash
taskset -c 0 perf stat \
-e cycles,instructions,\
L1-dcache-loads,L1-dcache-load-misses,\
l2_rqsts.demand_data_rd_hit,l2_rqsts.demand_data_rd_miss,\
mem_load_retired.l3_hit,mem_load_retired.l3_miss \
./cache_perf columnas
```

La región medida es:

```text
perf stat inicia
      │
      ▼
DISABLE
      │
      ├── malloc
      ├── inicialización
      ├── preparación
      │
      ▼
ENABLE
      │
      ▼
recorrido de matriz
      │
      ▼
DISABLE
      │
      ├── printf
      ├── free
      │
      ▼
fin
```


## 4. Resultados obtenidos

### Filas

```text
Matriz: 8000 x 8000
Elementos: 64000000
Memoria: 488.28 MiB

Modo: filas
Suma: 64000000.0
Tiempo recorrido: 0.148178 s

Performance counter stats for './cache_perf filas':

1,205,530,431  cycles:u
1,621,782,520  instructions:u                   # 1.35 insn per cycle
812,748,183    L1-dcache-loads:u
8,386,483      L1-dcache-load-misses:u          # 1.03%
299,418        l2_rqsts.demand_data_rd_hit:u
161,149        l2_rqsts.demand_data_rd_miss:u
2,455          mem_load_retired.l3_hit:u
123,047        mem_load_retired.l3_miss:u
```

### Columnas

```text
Matriz: 8000 x 8000
Elementos: 64000000
Memoria: 488.28 MiB

Modo: columnas
Suma: 64000000.0
Tiempo recorrido: 0.675199 s

Performance counter stats for './cache_perf columnas':

3,435,666,571  cycles:u
1,672,525,720  instructions:u                   # 0.49 insn per cycle
830,259,367    L1-dcache-loads:u
194,555,595    L1-dcache-load-misses:u          # 23.43%
83,199,043     l2_rqsts.demand_data_rd_hit:u
95,585,922     l2_rqsts.demand_data_rd_miss:u
52,581,411     mem_load_retired.l3_hit:u
11,377,421     mem_load_retired.l3_miss:u
```

> Nota: en esta ejecución `perf` mostró porcentajes de multiplexación menores de 100% para varios eventos. Por ello conviene repetir las mediciones por grupos de eventos.


## 5. Análisis comparativo

| Métrica | Filas | Columnas | Cambio |
|---|---:|---:|---:|
| Tiempo del recorrido | 0.148 s | 0.675 s | 4.56× más lento |
| Cycles | 1.206 G | 3.436 G | 2.85× más |
| Instructions | 1.622 G | 1.673 G | ~1.03× |
| IPC | 1.35 | 0.49 | caída fuerte |
| L1 miss rate | 1.03 % | 23.43 % | incremento muy grande |
| L2 miss rate | 34.99 % | 53.46 % | empeora |

### L2

Para filas:

$$
\text{MissRate}_{L2}
=
\frac{161149}{299418+161149}\times 100
\approx 34.99\%
$$

Para columnas:

$$
\text{MissRate}_{L2}
=
\frac{95585922}{83199043+95585922}\times 100
\approx 53.46\%
$$

### Interpretación

El recorrido por filas aprovecha la localidad espacial porque los elementos consecutivos de una fila están contiguos en memoria.

Para una matriz `8000 × 8000` de `double`, el salto entre dos elementos consecutivos de una columna es:
$$
8000 \times 8 = 64000\;\text{bytes}
$$

es decir:

$$
64000 / 1024 = 62.5\;\text{KiB}
$$

Por eso el patrón por columnas genera mucha más presión sobre la jerarquía de caché.

La relación observada es:

```text
mala localidad espacial
        ↓
más misses de caché
        ↓
más esperas por datos
        ↓
más ciclos
        ↓
menor IPC
        ↓
mayor tiempo de ejecución
```


## 6. Medición recomendada para evitar multiplexación

Al solicitar muchos eventos simultáneamente, `perf` puede multiplexar los contadores. Para obtener resultados más precisos, conviene separar las mediciones.

### CPU + L1

```bash
taskset -c 0 perf stat \
-e cycles,instructions,L1-dcache-loads,L1-dcache-load-misses \
./cache_perf filas
```

```bash
taskset -c 0 perf stat \
-e cycles,instructions,L1-dcache-loads,L1-dcache-load-misses \
./cache_perf columnas
```

### L2

```bash
taskset -c 0 perf stat \
-e l2_rqsts.demand_data_rd_hit,l2_rqsts.demand_data_rd_miss \
./cache_perf filas
```

```bash
taskset -c 0 perf stat \
-e l2_rqsts.demand_data_rd_hit,l2_rqsts.demand_data_rd_miss \
./cache_perf columnas
```

### L3

```bash
taskset -c 0 perf stat \
-e mem_load_retired.l3_hit,mem_load_retired.l3_miss \
./cache_perf filas
```

```bash
taskset -c 0 perf stat \
-e mem_load_retired.l3_hit,mem_load_retired.l3_miss \
./cache_perf columnas
```


## 7. Conclusión

Este experimento permite conectar directamente:

$$
\boxed{\text{patrón de acceso}}
\rightarrow
\boxed{\text{localidad espacial}}
\rightarrow
\boxed{\text{L1/L2/L3}}
\rightarrow
\boxed{\text{stalls}}
\rightarrow
\boxed{\text{IPC}}
\rightarrow
\boxed{\text{tiempo de ejecución}}
$$

En los resultados obtenidos, el recorrido por columnas tardó aproximadamente **4.56 veces más** que el recorrido por filas, mientras que el número total de instrucciones fue muy parecido. La degradación se explica principalmente por el aumento de los fallos de caché y la consecuente caída del IPC.
